# Ontario Electricity + FSA Temperature Data Pipeline

This notebook downloads and cleans hourly IESO electricity data, loads hourly temperature data by FSA, joins the datasets, creates calendar/weather features, and writes 48 hour/day-type CSV files.

## Before running

1. Ensure your temperature notebook has created `data/fsa_temperature_hourly.parquet` with `FSA`, a timestamp column (`time_est` or `time_utc`), and `temp_c`.
2. Update the date range and paths in the configuration cell.
3. Install dependencies if needed: `pip install pandas requests holidays pyarrow`.

**Important:** The IESO delivery-hour convention and temperature timezone must be checked against your source files, especially around daylight-saving transitions. The notebook currently preserves the friend's fixed-EST `time_est` convention when available.


## 1. Imports and configuration

In [ ]:
"""
Build Ontario FSA hourly electricity + temperature training data.

Outputs 48 CSV files:
  Business_Day_0100.csv ... Business_Day_2400.csv
  Not_Business_Day_0100.csv ... Not_Business_Day_2400.csv

Dependencies:
  pip install pandas requests holidays pyarrow

Expected weather input:
  data/fsa_temperature_hourly.parquet
  with columns FSA, time_est (or time_utc), temp_c.
The friend's notebook currently writes time_est using fixed UTC-5. This script
uses that column when available to stay consistent with the notebook.
"""

from __future__ import annotations

import io
import re
import time
import zipfile
from pathlib import Path

import holidays
import pandas as pd
import requests

# ================================ CONFIG ==================================== #
START_YEAR, START_MONTH = 2022, 1
END_YEAR, END_MONTH = 2023, 12  # inclusive

OUTPUT_DIR = Path("train_data")
WEATHER_PARQUET = Path("data/fsa_temperature_hourly.parquet")
CENTROIDS_CSV = None  # optional CSV with FSA,LAT,LON; otherwise no coordinates
LOCAL_ZIP_DIR = None  # optional folder containing downloaded IESO monthly ZIPs

KEEP_PRICE_PLANS = {"TIERED", "TOU"}
KEEP_CUSTOMER_TYPES = None  # e.g. {"RESIDENTIAL"}; None keeps every type
DROP_ROWS_WITHOUT_TEMPERATURE = True
PRINT_DISTINCT_VALUES = True
BASE_TEMP_C = 18.0
# ============================================================================ #

## 2. Helper functions
Run this cell to define download, parsing, cleaning, weather-loading, and output-splitting functions.

In [ ]:
BASE_URL = "https://reports-public.ieso.ca/public/HourlyConsumptionByFSA"
FSA_RE = re.compile(r"^[A-Z]\d[A-Z]$")

COLUMN_CANDIDATES = {
    "FSA": ["FSA", "FORWARD_SORTATION_AREA"],
    "DATE": ["DATE", "DELIVERY_DATE"],
    "HOUR": ["HOUR", "DELIVERY_HOUR"],
    "CUSTOMER_TYPE": ["CUSTOMER_TYPE", "CUSTOMERTYPE", "CUSTOMER_CLASS"],
    "PRICE_PLAN": ["PRICE_PLAN", "PRICEPLAN", "PRICE_PLAN_TYPE"],
    "TOTAL_CONSUMPTION": ["TOTAL_CONSUMPTION", "TOTAL_CONSUMPTION_KWH", "CONSUMPTION_KWH"],
    "PREMISE_COUNT": ["PREMISE_COUNT", "PREMISES", "NUMBER_OF_PREMISES"],
}

OUTPUT_ORDER = [
    "DATE", "HOUR", "FSA", "CUSTOMER_TYPE", "PRICE_PLAN",
    "TOTAL_CONSUMPTION", "PREMISE_COUNT", "CONSUMPTION_PER_UNIT",
    "IS_BUSINESS_DAY", "IS_WEEKDAY", "IS_HOLIDAY", "DAY_NAME",
    "YEAR", "MONTH", "WEEK_OF_YEAR", "DAY_OF_YEAR", "DAY_OF_WEEK",
    "REGION_LETTER", "IS_RURAL", "LAT", "LON",
    "TEMPERATURE", "HDD", "CDD",
]


def norm_col(value):
    return re.sub(r"[^A-Z0-9]+", "_", str(value).strip().upper()).strip("_")


def month_range(start_year, start_month, end_year, end_month):
    year, month = start_year, start_month
    while (year, month) <= (end_year, end_month):
        yield year, month
        month += 1
        if month == 13:
            year, month = year + 1, 1


def download_zip(year, month, dest_dir, retries=3):
    filename = f"PUB_HourlyConsumptionByFSA_{year}{month:02d}_v1.zip"
    path = Path(dest_dir) / filename
    if path.exists():
        return path

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(f"{BASE_URL}/{filename}", stream=True, timeout=120)
            if response.status_code == 404:
                print(f"  [skip] {filename} is not published")
                return None
            response.raise_for_status()
            with path.open("wb") as output:
                for chunk in response.iter_content(1024 * 1024):
                    if chunk:
                        output.write(chunk)
            return path
        except requests.RequestException as exc:
            print(f"  [retry {attempt}/{retries}] {exc}")
            time.sleep(2 * attempt)

    print(f"  [fail] Could not download {filename}")
    return None


def read_ieso_zip(path):
    """Read CSVs in a monthly ZIP, detecting the real header after any preamble."""
    frames = []
    with zipfile.ZipFile(path) as archive:
        for member in archive.namelist():
            if not member.lower().endswith(".csv"):
                continue
            content = archive.read(member).decode("utf-8-sig", errors="replace")
            lines = content.splitlines()
            header_line = next(
                (
                    index for index, line in enumerate(lines[:40])
                    if "FSA" in line.upper()
                    and "DATE" in line.upper()
                    and line.count(",") >= 3
                ),
                None,
            )
            if header_line is None:
                raise ValueError(f"Could not find FSA/DATE header in {member}")
            frames.append(pd.read_csv(io.StringIO(content), skiprows=header_line, dtype=str))

    if not frames:
        raise ValueError(f"No CSV files found in {path}")
    return pd.concat(frames, ignore_index=True)


def standardise_columns(df):
    df = df.rename(columns={column: norm_col(column) for column in df.columns})
    rename = {}
    for canonical, candidates in COLUMN_CANDIDATES.items():
        found = next((candidate for candidate in candidates if candidate in df.columns), None)
        if found is None:
            raise KeyError(
                f"Missing required column {canonical}. Found columns: {list(df.columns)}"
            )
        rename[found] = canonical
    return df.rename(columns=rename)


def load_weather_month(path, start_date, end_date):
    """
    Read only the requested month's weather rows where PyArrow can apply filters.
    time_est timestamps are treated as the local timestamp used by the friend's
    notebook. IESO HOUR 1 maps to timestamp hour 00:00; HOUR 24 maps to 23:00.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"Temperature parquet not found: {path}. Run the temperature notebook first."
        )

    try:
        import pyarrow.dataset as ds
    except ImportError as exc:
        raise ImportError("Install pyarrow: pip install pyarrow") from exc

    dataset = ds.dataset(str(path), format="parquet")
    available = set(dataset.schema.names)
    time_col = "time_est" if "time_est" in available else "time_utc"
    required = {"FSA", time_col}
    if not required.issubset(available):
        raise ValueError(
            f"Weather parquet must contain FSA and time_est or time_utc. Found: {sorted(available)}"
        )

    temp_col = next((c for c in ("temp_c", "TEMPERATURE", "temperature") if c in available), None)
    if temp_col is None:
        raise ValueError("Weather parquet needs a temperature column such as temp_c.")

    # Include the full final day; end_date is exclusive.
    filt = (ds.field(time_col) >= pd.Timestamp(start_date).to_pydatetime()) & (
        ds.field(time_col) < pd.Timestamp(end_date).to_pydatetime()
    )
    table = dataset.to_table(columns=["FSA", time_col, temp_col], filter=filt)
    weather = table.to_pandas()
    if weather.empty:
        return pd.DataFrame(columns=["FSA", "DATE", "HOUR", "TEMPERATURE"])

    weather["FSA"] = weather["FSA"].astype(str).str.strip().str.upper()
    weather[time_col] = pd.to_datetime(weather[time_col], errors="coerce")
    weather["TEMPERATURE"] = pd.to_numeric(weather[temp_col], errors="coerce").astype("float32")

    # ERA5 timestamps mark the beginning of the hour; IESO HOUR is 1..24.
    weather = weather.dropna(subset=[time_col, "TEMPERATURE"])
    weather["DATE"] = weather[time_col].dt.normalize()
    weather["HOUR"] = weather[time_col].dt.hour + 1
    weather = weather[weather["FSA"].str.match(FSA_RE)]
    weather = weather[["FSA", "DATE", "HOUR", "TEMPERATURE"]]

    duplicates = weather.duplicated(["FSA", "DATE", "HOUR"])
    if duplicates.any():
        # Same FSA-hour should have one value; averaging protects against duplicate
        # records without multiplying electricity rows during the merge.
        weather = weather.groupby(
            ["FSA", "DATE", "HOUR"], as_index=False, observed=True
        )["TEMPERATURE"].mean()

    return weather


def load_centroids(path):
    if path is None:
        return None
    coords = pd.read_csv(path, dtype={"FSA": str})
    coords.columns = [norm_col(c) for c in coords.columns]
    required = {"FSA", "LAT", "LON"}
    if not required.issubset(coords.columns):
        raise ValueError(f"Centroid CSV must have columns FSA,LAT,LON; got {list(coords.columns)}")
    coords["FSA"] = coords["FSA"].str.strip().str.upper()
    return coords[["FSA", "LAT", "LON"]].drop_duplicates("FSA")


def clean_electricity(raw, ontario_holidays):
    df = standardise_columns(raw)
    df["FSA"] = df["FSA"].astype(str).str.strip().str.upper()
    df["PRICE_PLAN"] = df["PRICE_PLAN"].astype(str).str.strip().str.upper()
    df["CUSTOMER_TYPE"] = df["CUSTOMER_TYPE"].astype(str).str.strip().str.upper()

    df = df[df["FSA"].str.match(FSA_RE) & df["PRICE_PLAN"].isin(KEEP_PRICE_PLANS)]
    if KEEP_CUSTOMER_TYPES is not None:
        keep_types = {value.upper() for value in KEEP_CUSTOMER_TYPES}
        df = df[df["CUSTOMER_TYPE"].isin(keep_types)]

    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce").dt.normalize()
    for column in ("HOUR", "TOTAL_CONSUMPTION", "PREMISE_COUNT"):
        df[column] = pd.to_numeric(df[column], errors="coerce")
    df = df.dropna(subset=["DATE", "HOUR", "TOTAL_CONSUMPTION", "PREMISE_COUNT"])
    df = df[df["HOUR"].between(1, 24) & (df["PREMISE_COUNT"] > 0)].copy()

    df["HOUR"] = df["HOUR"].astype("int8")
    df["TOTAL_CONSUMPTION"] = df["TOTAL_CONSUMPTION"].astype("float32")
    df["PREMISE_COUNT"] = df["PREMISE_COUNT"].astype("int32")
    df["CONSUMPTION_PER_UNIT"] = (
        df["TOTAL_CONSUMPTION"] / df["PREMISE_COUNT"]
    ).astype("float32")

    df["DAY_NAME"] = df["DATE"].dt.day_name()
    df["DAY_OF_WEEK"] = df["DATE"].dt.dayofweek.astype("int8")  # Monday=0, Sunday=6
    df["IS_WEEKDAY"] = (df["DAY_OF_WEEK"] < 5).astype("int8")
    df["IS_HOLIDAY"] = df["DATE"].isin(ontario_holidays).astype("int8")
    df["IS_BUSINESS_DAY"] = (
        (df["IS_WEEKDAY"] == 1) & (df["IS_HOLIDAY"] == 0)
    ).astype("int8")
    df["YEAR"] = df["DATE"].dt.year.astype("int16")
    df["MONTH"] = df["DATE"].dt.month.astype("int8")
    df["WEEK_OF_YEAR"] = df["DATE"].dt.isocalendar().week.astype("int8")
    df["DAY_OF_YEAR"] = df["DATE"].dt.dayofyear.astype("int16")
    df["REGION_LETTER"] = df["FSA"].str[0]
    df["IS_RURAL"] = (df["FSA"].str[1] == "0").astype("int8")
    return df


def append_split_files(df, out_dir):
    business = df["IS_BUSINESS_DAY"].eq(1)
    for (is_business, hour), group in df.groupby([business, "HOUR"], sort=False, observed=True):
        prefix = "Business_Day" if bool(is_business) else "Not_Business_Day"
        destination = out_dir / f"{prefix}_{int(hour):02d}00.csv"
        columns = [column for column in OUTPUT_ORDER if column in group.columns]
        group.loc[:, columns].to_csv(
            destination,
            mode="a",
            index=False,
            header=not destination.exists(),
            date_format="%Y-%m-%d",
        )



## 3. Run the pipeline
This clears only existing `Business_Day_*.csv` and `Not_Business_Day_*.csv` files in the output directory, then rebuilds the split files.

In [ ]:
def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    # Remove only files this script owns, so reruns cannot duplicate old rows.
    for old_file in list(OUTPUT_DIR.glob("Business_Day_*.csv")) + list(
        OUTPUT_DIR.glob("Not_Business_Day_*.csv")
    ):
        old_file.unlink()

    years = range(START_YEAR, END_YEAR + 1)
    holiday_dates = {
        pd.Timestamp(day)
        for day in holidays.Canada(subdiv="ON", years=years).keys()
    }
    centroids = load_centroids(CENTROIDS_CSV)

    total_rows = 0
    printed_values = False
    for year, month in month_range(START_YEAR, START_MONTH, END_YEAR, END_MONTH):
        print(f"\nProcessing {year}-{month:02d}")
        if LOCAL_ZIP_DIR:
            zip_path = Path(LOCAL_ZIP_DIR) / (
                f"PUB_HourlyConsumptionByFSA_{year}{month:02d}_v1.zip"
            )
        else:
            # Keep downloaded ZIPs in a cache to avoid downloading again next run.
            cache_dir = OUTPUT_DIR / "_ieso_zip_cache"
            cache_dir.mkdir(parents=True, exist_ok=True)
            zip_path = download_zip(year, month, cache_dir)

        if zip_path is None or not Path(zip_path).exists():
            continue

        raw = read_ieso_zip(zip_path)
        if PRINT_DISTINCT_VALUES and not printed_values:
            standard = standardise_columns(raw)
            print("  PRICE_PLAN values:", sorted(standard["PRICE_PLAN"].astype(str).unique()))
            print("  CUSTOMER_TYPE values:", sorted(standard["CUSTOMER_TYPE"].astype(str).unique()))
            printed_values = True

        electricity = clean_electricity(raw, holiday_dates)
        if electricity.empty:
            print("  No electricity rows after filtering.")
            continue

        if centroids is not None:
            electricity = electricity.merge(centroids, on="FSA", how="left", validate="many_to_one")

        month_start = pd.Timestamp(year=year, month=month, day=1)
        month_end = month_start + pd.offsets.MonthBegin(1)
        weather = load_weather_month(WEATHER_PARQUET, month_start, month_end)

        before = len(electricity)
        electricity = electricity.merge(
            weather,
            on=["FSA", "DATE", "HOUR"],
            how="left",
            validate="many_to_one",
        )
        missing_temp = electricity["TEMPERATURE"].isna().sum()
        print(
            f"  electricity rows: {before:,} | missing temperature: "
            f"{missing_temp:,} ({missing_temp / max(before, 1):.1%})"
        )

        if DROP_ROWS_WITHOUT_TEMPERATURE:
            electricity = electricity.dropna(subset=["TEMPERATURE"]).copy()

        electricity["HDD"] = (BASE_TEMP_C - electricity["TEMPERATURE"]).clip(lower=0)
        electricity["CDD"] = (electricity["TEMPERATURE"] - BASE_TEMP_C).clip(lower=0)

        append_split_files(electricity, OUTPUT_DIR)
        total_rows += len(electricity)
        print(f"  rows written: {len(electricity):,}")

    output_files = list(OUTPUT_DIR.glob("Business_Day_*.csv")) + list(
        OUTPUT_DIR.glob("Not_Business_Day_*.csv")
    )
    print(f"\nFinished: {total_rows:,} rows written across {len(output_files)} split files.")
    print(f"Output folder: {OUTPUT_DIR.resolve()}")
    if len(output_files) != 48:
        print(
            "WARNING: expected 48 output files. Some may be absent if a particular "
            "hour/day type has no observations in the selected date range."
        )


if __name__ == "__main__":
    main()